---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 3.2: Add more tools

## 💬 Product Check-in:

>
> **Sam:** "Hey — a few users have mentioned that code examples from the agent don't always run. Syntax errors, missing indentation, that kind of thing. Not a huge volume but it's coming up enough to be worth addressing."
>

**Objective:** Translate next round of feedback into evaluations and scorers that can be used to refine the agent. Refine aspects of the agent to meet updated evaluation criteria.

In this notebook:

1. Create code validation tool, test outside of agent
2. Add tool to agent
3. Evaluate agent with new tool, check for performance improvements.

---

In [ ]:
from functools import cached_property
from pathlib import Path
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ENV_FILE = Path.cwd().parent.parent / ".env"  # adjust .parent count to match your nesting

class AgentConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",  # don't choke on unrelated vars in .env
    )

    # Secrets / endpoints (from .env)
    gemini_api_key: SecretStr
    gemini_openai_base_url: str

    # MLflow (from .env, with fallbacks)
    mlflow_tracking_uri: str
    experiment_name: str = Field(default="mlflow-agent-1", alias="EXPERIMENT_1_NAME")

    # Model knobs (hardcoded defaults, overridable via env if you want)
    model: str = "gemini-2.5-flash-lite"
    temperature: float = 0.2
    max_tokens: int = 512
    ## Adding judge model to config
    judge_model: str = "gemini:/gemini-3.1-flash-lite-preview"

    # Prompt
    prompt_name: str = "mlflow-agent-system"
    system_prompt: str = "You are an MLflow assistant."

    ##DATASET
    eval_dataset_name: str = "mlflow-agent-eval"

    # Aligned Judge
    align_judge: str = "response_quality"

    @cached_property
    def prompt_alias(self) -> str:
        return f"prompts:/{self.prompt_name}@prod"

cfg = AgentConfig()

In [ ]:
import mlflow

mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.experiment_name)
mlflow.langchain.autolog()

prompt_version = mlflow.genai.load_prompt(cfg.prompt_alias)
SYSTEM_PROMPT = prompt_version.template

In [ ]:
import ast

def validate_python_syntax(code: str) -> dict:
    """
    Validates Python syntax using the built-in ast module.
    No external dependencies required.

    Args:
        code: A string of Python code to validate.

    Returns:
        A dict with keys: valid, error, line, offset
    """
    try:
        ast.parse(code)
        return {
            "valid": True,
            "error": None,
            "line": None,
            "offset": None,
        }
    except SyntaxError as e:
        return {
            "valid": False,
            "error": str(e.msg),
            "line": e.lineno,
            "offset": e.offset,
        }

In [ ]:
# Test tool
validate_python_syntax("def foo(x):\n    return x + 1")

In [ ]:
validate_python_syntax("public static void main(String[] args) {\n    System.out.println('Hello, world!');\n}")

In [ ]:
# Create Agent
from tool_agent import agent

def predict_fn(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [ ]:
predict_fn("What is MLflow?")

In [ ]:
code_quality_evals = [
    {
        # Only one right way to call this — specific args make the expected response tight
        "inputs": {"question": "Show me how to log a metric called 'accuracy' with a value of 0.95 at step 1 in MLflow."},
        "expectations": {
            "expected_response": "mlflow.log_metric('accuracy', 0.95, step=1)",
        },
    },
    {
        # Forces a specific parameter name and value — no ambiguity
        "inputs": {"question": "Show me how to log a parameter called 'learning_rate' with a value of 0.01 in MLflow."},
        "expectations": {
            "expected_response": "mlflow.log_param('learning_rate', 0.01)",
        },
    },
    {
        # Specific enough that the correct answer is a single line
        "inputs": {"question": "Show me how to set an experiment called 'my_experiment' in MLflow."},
        "expectations": {
            "expected_response": "mlflow.set_experiment('my_experiment')",
        },
    },
    {
        # Specific enough that the correct answer is a single line
        "inputs": {"question": "Show me how to search for runs in an experiment called 'my_experiment' in MLflow."},
        "expectations": {
            "expected_response": "mlflow.search_runs(experiment_names=['my_experiment'])",
        },
    },
]
print(f"Code quality eval set size: {len(code_quality_evals)} examples")

# Add examples to full dataset

In [ ]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
    ExpectationsGuidelines
)

results = mlflow.genai.evaluate(
    data=code_quality_evals,
    predict_fn=predict_fn,
    scorers=[
        ToolCallCorrectness(model=cfg.judge_model), 
        ToolCallEfficiency(model=cfg.judge_model), 
        Correctness(model=cfg.judge_model),
        RelevanceToQuery(model=cfg.judge_model),
        ExpectationsGuidelines(model=cfg.judge_model)
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)

In [ ]:
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name=cfg.eval_dataset_name)

In [ ]:
retrieved_dataset.merge_records(code_quality_evals)

In [ ]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
    ExpectationsGuidelines
)

results = mlflow.genai.evaluate(
    data=retrieved_dataset,
    predict_fn=predict_fn,
    scorers=[
        ToolCallCorrectness(model=cfg.judge_model), 
        ToolCallEfficiency(model=cfg.judge_model), 
        Correctness(model=cfg.judge_model),
        RelevanceToQuery(model=cfg.judge_model),
        ExpectationsGuidelines(model=cfg.judge_model)
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)